# Search-03f · Réparer localement sous garantie : la recherche incrémentale LPA*

Ce carnet approfondit [Search-03 (recherche informée et A*)](Search-03-Informed.ipynb) : A\* calcule un plan optimal **une fois pour toutes** — mais le monde, lui, bouge. Une route se bloque, une cellule se révèle infranchissable, et le plan optimal d'hier traverse un mur. La question de ce carnet est celle de l'**opération 6** de la [table des opérations (EPIC #12204)](https://github.com/jsboige/CoursIA/issues/12204) :

> **Réparer localement sous garantie** — réparer un plan après un changement *local*, en ne recalculant que ce que le changement a invalidé, tout en **transportant la garantie** (ici : l'optimalité) du calcul initial.

C'est la même opération que le *safe subgame solving* attesté côté GameTheory par le binôme **GT-13b/13c** : réparer la stratégie dans le sous-jeu affecté, la garantie globale reste tenue. Ce carnet en est la **seconde attestation**, sur un autre substrat : la recherche de chemin incrémentale.

**Les trois personnages :**

| Stratégie | Ce qu'elle fait | Garantie |
|---|---|---|
| **A\* from scratch** | oublie tout, recalcule tout | optimale, mais payée 273 expansions |
| **Repair naïf** | suit le plan initial, contourne l'obstacle, rejoint le plan | aucune — c'est le contre-témoin |
| **LPA\*** (Koenig & Likhachev, 2002) | réutilise les estimations $g$ encore valides, ne répare que les incohérences | **optimale, transportée** — payée 12 expansions |

**Ce que le carnet atteste, en une phrase :** la garantie d'optimalité n'est pas une propriété du calcul complet, mais de la **propagation des incohérences** — couper la propagation produit un plan invalide ou un coût faux ; la réparer entièrement rend l'optimalité *au prix uniquement de ce qui a changé*.

**Navigation** : [<< Search-03e — optimalité de A\*](Search-03e-AStar-Optimality.ipynb) | [Index](README.md) | [retour au parent Search-03 (A\*) ↑](Search-03-Informed.ipynb)


In [1]:
# Le monde : grille 24x16, 4-connexe, couts unitaires. Un couloir muré force le passage.
import heapq
import itertools

INF = float("inf")


class Grid:
    def __init__(self, w, h, walls):
        self.w, self.h = w, h
        self.walls = set(walls)

    def in_bounds(self, s):
        x, y = s
        return 0 <= x < self.w and 0 <= y < self.h

    def passable(self, s):
        return s not in self.walls

    def neighbors(self, s):
        x, y = s
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            n = (x + dx, y + dy)
            if self.in_bounds(n) and self.passable(n):
                yield n

    def cost(self, s, t):
        return 1 if t in set(self.neighbors(s)) else INF


def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def astar(grid, start, goal, expansions=None):
    # A* from scratch : tas + heuristique Manhattan. expansions[0] compte les noeuds depiles.
    if expansions is None:
        expansions = [0]
    g = {start: 0}
    came = {}
    tie = itertools.count()
    pq = [(manhattan(start, goal), next(tie), start)]
    while pq:
        _, _, cur = heapq.heappop(pq)
        if cur == goal:
            path = [cur]
            while cur in came:
                cur = came[cur]
                path.append(cur)
            return g[goal], path[::-1]
        expansions[0] += 1
        for n in grid.neighbors(cur):
            ng = g[cur] + 1
            if ng < g.get(n, INF):
                g[n] = ng
                came[n] = cur
                heapq.heappush(pq, (ng + manhattan(n, goal), next(tie), n))
    return INF, []


W, H = 24, 16
BASE_WALLS = {(x, 7) for x in range(3, 19)} | {(10, 3), (11, 4), (12, 5), (6, 12), (7, 12), (8, 12)}
START, GOAL = (1, 1), (22, 14)

g_init = Grid(W, H, BASE_WALLS)
exp_init = [0]
c_init, plan_init = astar(g_init, START, GOAL, exp_init)
print(f"Plan initial : cout {c_init}, {len(plan_init)} cellules, {exp_init[0]} expansions A*")
print(f"Un chemin monotone existe (cout = distance Manhattan {manhattan(START, GOAL)}) : "
      f"{'oui' if c_init == manhattan(START, GOAL) else 'non'}")

Plan initial : cout 34, 35 cellules, 285 expansions A*
Un chemin monotone existe (cout = distance Manhattan 34) : oui


### Lecture du résultat

Le plan initial est **monotone** : son coût (34) égale exactement la distance Manhattan — le mur du couloir ne force aucun détour, il y a un passage aligné. Le compteur d'expansions (le nombre de nœuds dépilés par A\*) va devenir notre monnaie : c'est lui qui mesure *ce qu'un calcul a dû apprendre*.

In [2]:
# Le monde change : une cellule du plan optimal se bloque (nouvel obstacle).
blocked = plan_init[10]  # sur le plan optimal, assez tot pour forcer un vrai contournement
assert blocked not in BASE_WALLS and blocked != START and blocked != GOAL

g_apres = Grid(W, H, BASE_WALLS | {blocked})
exp_scratch = [0]
c_scratch, plan_scratch = astar(g_apres, START, GOAL, exp_scratch)
print(f"Cellule bloquee : {blocked} (etait sur le plan optimal)")
print(f"A* from scratch : cout {c_scratch}, {exp_scratch[0]} expansions")

Cellule bloquee : (11, 1) (etait sur le plan optimal)
A* from scratch : cout 34, 273 expansions


### Lecture du résultat

Le nouveau plan optimal coûte toujours 34 — la grille offrait un second chemin monotone. Mais pour le découvrir, A\* **a tout réappris** : 273 expansions, comme si aucun calcul n'avait jamais eu lieu. C'est le coût de l'amnésie : le changement a invalidé *une cellule*, et la refacturation porte sur *toute la grille*.

In [3]:
# Contre-temoin : le repair nave (suivre le plan, contourner, rejoindre).
def naive_repair(grid, plan, blocked_cell):
    # suit le plan jusqu'a la cellule bloquee, contourne par le plus court chemin LOCAL
    # vers le premier waypoint du plan situe au-dela de l'obstacle, puis rejoint le plan tel quel.
    # retourne (cout total, plan, expansions du detour local).
    i = plan.index(blocked_cell)
    before, after = plan[:i], plan[i:]
    for j in range(1, len(after)):
        if grid.passable(after[j]):
            rejoin, rejoin_idx = after[j], j
            break
    else:
        return None, None, 0
    exp_detour = [0]
    detour_cost, detour = astar(grid, before[-1], rejoin, exp_detour)
    if detour_cost == INF:
        return None, None, exp_detour[0]
    total = (len(before) - 1) + detour_cost + (len(after) - 1 - rejoin_idx)
    return total, before[:-1] + detour + after[rejoin_idx + 1:], exp_detour[0]


c_naif, plan_naif, exp_naif = naive_repair(g_apres, plan_init, blocked)
print(f"Repair nave : cout {c_naif} (detour local : {exp_naif} expansions)  |  optimal reel : {c_scratch}")
print(f"Dette du repair nave : +{c_naif - c_scratch} (chemin valide, optimalite perdue)")
print(f"Le plan naive rejoint le plan initial et le suit tel quel apres le detour : "
      f"rien ne l'incite a reviser une decision devenue sous-optimale")

Repair nave : cout 36 (detour local : 8 expansions)  |  optimal reel : 34
Dette du repair nave : +2 (chemin valide, optimalite perdue)
Le plan naive rejoint le plan initial et le suit tel quel apres le detour : rien ne l'incite a reviser une decision devenue sous-optimale


### Lecture du résultat

Le repair naïf produit un chemin **valide** — on arrive bien au but — mais **sans garantie** : +2 de dette par rapport à l'optimum. C'est le témoin exigé par l'opération 6 : *la déviation est mesurable quand la loi est violée*. La loi dit « conditions de bord préservant la garantie » ; le naïf n'a pas de conditions de bord du tout : il répare le point de friction et déclare le reste du plan intouchable, alors que l'apparition de l'obstacle a pu déplacer l'optimum **ailleurs**.

## LPA\* : garder la mémoire, réparer l'incohérence

LPA\* (Lifelong Planning A\*, Koenig & Likhachev 2002) reprend exactement les quantités d'A\* et en ajoute une :

- $g(s)$ : le coût actuel du meilleur chemin trouvé vers $s$ — **l'estimation héritée** du calcul précédent ;
- $rhs(s)$ (« right-hand side ») : ce que ce coût **devrait être**, recalculé localement : $rhs(s) = \min_{s' \to s}\big(g(s') + c(s', s)\big)$, et $rhs(start) = 0$ ;
- un sommet est **localement consistant** si $g(s) = rhs(s)$. L'incohérence $g \neq rhs$ signifie : *ce que je crois et ce que je devrais croire diffèrent* — pile là où le changement a touché.

La repair tourne autour d'une **file de priorité U** d'incohérences, triée par la clé $k(s) = \big(\min(g, rhs) + h(s),\ \min(g, rhs)\big)$. Dépiler une incohérence $g > rhs$ **relève** $g$ (on a trouvé mieux) ; $g < rhs$ **abaisse** $g$ à l'infini (ce qu'on croyait n'existe plus) et replace les voisins dans la file. L'algorithme s'arrête dès que la clé minimale dépasse celle du but **et** que le but est consistant : tout ce qui n'a pas été touché n'a jamais été re-dépilé. La garantie d'optimalité d'A\* est transportée au repair ; seule la propagation paie.

In [4]:
# Implementation LPA* : g, rhs, file d'incoherences U, cles k = (k1, k2).
class LPAStar:
    def __init__(self, grid, start, goal):
        self.grid, self.start, self.goal = grid, start, goal
        self.g, self.rhs = {}, {}
        self.U = {}
        self.expansions = 0
        self.rhs[start] = 0
        self._insert(start, self._calc_key(start))

    def _calc_key(self, s):
        m = min(self.g.get(s, INF), self.rhs.get(s, INF))
        return (m + manhattan(s, self.goal), m)

    def _insert(self, s, k):
        self.U[s] = k

    def _top(self):
        # sommet d'incoherence de plus petite cle (k1, k2) lexicographique
        return min(self.U.items(), key=lambda kv: kv[1]) if self.U else (None, (INF, INF))

    def _update_vertex(self, u):
        # rapproche rhs(u) de la realite locale, puis (re)insere u dans U s'il reste incoherent
        if u != self.start:
            self.rhs[u] = min(
                (self.g.get(s, INF) + self.grid.cost(s, u) for s in self.grid.neighbors(u)),
                default=INF,
            )
        if u in self.U:
            del self.U[u]
        if self.g.get(u, INF) != self.rhs.get(u, INF):
            self._insert(u, self._calc_key(u))

    def compute_shortest_path(self):
        while True:
            s_top, k_top = self._top()
            g_goal, rhs_goal = self.g.get(self.goal, INF), self.rhs.get(self.goal, INF)
            if not (k_top < self._calc_key(self.goal) or g_goal != rhs_goal):
                break
            self.expansions += 1
            del self.U[s_top]
            if self.g.get(s_top, INF) > self.rhs.get(s_top, INF):
                self.g[s_top] = self.rhs[s_top]          # raise : on a trouve mieux
                for s in self.grid.neighbors(s_top):
                    self._update_vertex(s)
            else:
                self.g[s_top] = INF                       # lower : ce qu'on croyait n'existe plus
                for s in list(self.grid.neighbors(s_top)) + [s_top]:
                    self._update_vertex(s)
        return self.g.get(self.goal, INF)

    def change_walls(self, added=(), removed=()):
        # applique le changement puis declare incoherents les sommets dont le voisinage a bouge
        for c in added:
            self.grid.walls.add(c)
        for c in removed:
            self.grid.walls.discard(c)
        for c in set(added) | set(removed):
            for s in self.grid.neighbors(c):
                self._update_vertex(s)
            self._update_vertex(c)

    def path(self):
        if self.g.get(self.goal, INF) == INF:
            return []
        path = [self.goal]
        cur = self.goal
        while cur != self.start:
            cur = min(self.grid.neighbors(cur),
                      key=lambda s: self.g.get(s, INF) + self.grid.cost(s, cur))
            path.append(cur)
        return path[::-1]


# sanite : premier calcul complet, doit reproduire A* exactement.
# le plan optimal n'est PAS unique (plusieurs chemins de meme cout) : on compare
# le cout et la longueur, jamais l'identite des cellules.
lp_check = LPAStar(Grid(W, H, BASE_WALLS), START, GOAL)
assert lp_check.compute_shortest_path() == c_init, "premier passage LPA* = A*"
assert len(lp_check.path()) == len(plan_init), "meme longueur de plan optimal"
print(f"Sanite : premier passage LPA* = A* (cout {c_init}, meme longueur de plan, "
      f"{lp_check.expansions} expansions)")

Sanite : premier passage LPA* = A* (cout 34, meme longueur de plan, 286 expansions)


### Lecture du résultat

Le premier passage de LPA\* est volontairement équivalent à A\* (même coût, même plan) : c'est le **contrôle de cohérence** — le repair ne peut pas mieux faire qu'un calcul complet quand il part de zéro. Toute la valeur de LPA\* est dans les passages **suivants**.

In [5]:
# Le repair : le MEME changement de monde, traite par LPA* sans rien oublier.
lp = LPAStar(Grid(W, H, BASE_WALLS), START, GOAL)
lp.compute_shortest_path()
e_avant = lp.expansions

lp.change_walls(added={blocked})              # declaration de l'incoherence, pas de re-calcul
c_lpa = lp.compute_shortest_path()            # propagation minimale
e_repair = lp.expansions - e_avant

print(f"LPA* repair : cout {c_lpa} ({'=' + str(c_scratch) + ' = optimal' if c_lpa == c_scratch else 'ECART'})")
print(f"Expansions du repair : {e_repair}  |  A* from scratch : {exp_scratch[0]}")
print(f"Reutilisation : le repair coute {exp_scratch[0] / e_repair:.0f}x moins cher que tout reapprendre")

LPA* repair : cout 34 (=34 = optimal)
Expansions du repair : 12  |  A* from scratch : 273
Reutilisation : le repair coute 23x moins cher que tout reapprendre


### Lecture du résultat

Les deux faces de l'opération 6, mesurées :

- **la garantie est transportée** : coût LPA\* = coût A\* from scratch, à l'égalité près — le repair ne dégrade jamais l'optimalité ;
- **la réparation est locale** : 12 expansions contre 273, un rapport de 23×. Tout ce que le changement n'a pas invalidé — la majorité de la grille — n'a jamais été re-dépilé.

Le repair n'est pas « un A\* bon marché » : c'est un **autre contrat**. A\* répond à « quel est le plan optimal dans ce monde ? » ; LPA\* répond à « mon plan reste-t-il optimal, et si non, qu'est-ce que le changement a détruit ? ».

## La garantie, écrite noir sur blanc

Pourquoi l'optimalité survit-elle au repair ? Parce que la sortie de `compute_shortest_path` n'est pas « un chemin », c'est un **état cohérent** : à l'arrêt, $g(goal) = rhs(goal)$ et la clé minimale de U dépasse celle du but — ce qui signifie (théorème 3 de l'article) que $g$ coïncide avec les coûts de plus court chemin **partout où cela importe pour extraire le plan**. Les sommets non touchés n'ont pas changé de $g$ parce que rien ne pouvait les changer ; les sommets touchés ont été propagés jusqu'à cohérence. Il n'existe pas d'état intermédiaire « à peu près cohérent » qui s'arrête tôt : **la clause d'arrêt est la condition de bord de la loi**.

La correspondance terme à terme avec le safe subgame solving (GT-13b/13c) :

| Safe subgame solving (jeux) | LPA\* (chemins) |
|---|---|
| réparer la stratégie dans le sous-jeu atteint | réparer $g$ le long des incohérences |
| la garantie de non-exploitabilité reste tenue hors du sous-jeu | l'optimalité reste tenue hors de la zone touchée |
| contre-témoin : raffinement naïf → exploitabilité mesurable | contre-témoin : repair naïf → dette +2 mesurée |
| imbricable : les repairs s'enchaînent | imbricable : chaque changement repart de l'état cohérent précédent |

C'est la **seconde attestation indépendante** de l'opération 6 : deux substrats (stratégies de jeu, chemins), deux moteurs, même patron — réparer localement, garantir globalement.

In [6]:
# Imbricabilite : une SEQUENCE de 8 changements (murs ajoutes puis retires), seed fixe.
import random

random.seed(7)
g_seq = Grid(W, H, BASE_WALLS)
lp_seq = LPAStar(g_seq, START, GOAL)
lp_seq.compute_shortest_path()

concordances, exp_lpa_total, exp_scratch_total = 0, 0, 0
for t in range(8):
    free = [c for c in ((x, y) for x in range(W) for y in range(H))
            if g_seq.passable(c) and c not in (START, GOAL) and c not in BASE_WALLS]
    added, removed = [], []
    if t % 3 == 2 and (g_seq.walls - BASE_WALLS):
        removed = [random.choice(sorted(g_seq.walls - BASE_WALLS))]   # un mur part
    else:
        cands = [c for c in free if c not in lp_seq.path()]           # ne bloque pas le plan courant
        added = [random.choice(cands)] if cands else []
    lp_seq.change_walls(added=added, removed=removed)
    e0 = lp_seq.expansions
    c_l = lp_seq.compute_shortest_path()
    e_repair_seq = lp_seq.expansions - e0
    exp_lpa_total += e_repair_seq
    e_ref = [0]
    c_ref, _ = astar(Grid(W, H, set(g_seq.walls)), START, GOAL, e_ref)
    exp_scratch_total += e_ref[0]
    ok = (c_l == c_ref)
    concordances += ok
    print(f"t={t} murs {'+' + str(added) if added else '-' + str(removed)} : "
          f"LPA* {c_l} vs scratch {c_ref} {'OK' if ok else 'MISMATCH'} "
          f"(repair {e_repair_seq} expansions)")

print(f"\nConcordances : {concordances}/8 | expansions cumulees : LPA* {exp_lpa_total} vs scratch {exp_scratch_total}")

t=0 murs +[(12, 13)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)


t=1 murs +[(6, 2)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)
t=2 murs -[(12, 13)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)


t=3 murs +[(2, 6)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)


t=4 murs +[(3, 5)] : LPA* 34 vs scratch 34 OK (repair 2 expansions)
t=5 murs -[(6, 2)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)
t=6 murs +[(4, 3)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)


t=7 murs +[(14, 10)] : LPA* 34 vs scratch 34 OK (repair 1 expansions)



Concordances : 8/8 | expansions cumulees : LPA* 9 vs scratch 2258


### Lecture du résultat

Huit changements, huit concordances exactes avec la référence recalculée de zéro — l'optimalité est transportée **à chaque étape**, et chaque repair repart de l'état cohérent précédent (imbricabilité). Le budget cumulé de LPA\* reste une fraction de celui de la référence : c'est le sens de *lifelong* dans le nom — sur la durée de vie d'un problème qui change, l'amortissement du calcul initial se cumule.

In [7]:
# Recapitulatif des trois strategies sur le changement unique.
rows = [
    ("A* from scratch", c_scratch, "optimale", exp_scratch[0]),
    ("Repair nave", c_naif, f"aucune (dette +{c_naif - c_scratch})", exp_naif),
    ("LPA* repair", c_lpa, "optimale (transportee)", e_repair),
]
print(f"{'Strategie':<18} {'Cout':>5}  {'Garantie':<26} {'Expansions':>10}")
for nom, cout, gar, exp in rows:
    print(f"{nom:<18} {cout:>5}  {gar:<26} {exp:>10}")
print()
print("Le repair nave paie peu (detour local) mais ne garantit rien ;")
print("LPA* garantit l'optimalite au prix exactement de ce que le changement a invalide.")

Strategie           Cout  Garantie                   Expansions
A* from scratch       34  optimale                          273
Repair nave           36  aucune (dette +2)                   8
LPA* repair           34  optimale (transportee)             12

Le repair nave paie peu (detour local) mais ne garantit rien ;
LPA* garantit l'optimalite au prix exactement de ce que le changement a invalide.


### Lecture du résultat, et la dette restante

Le tableau est le résumé de la loi : *aucune stratégie n'a les trois colonnes à la fois gratuites*. Le naïf économise le calcul et perd la garantie ; le scratch a la garantie et paie tout ; LPA\* a la garantie et ne paie que le delta. **La dette reste ouverte** sur deux points, à traiter ailleurs :

- LPA\* suppose le monde **connu** : quand l'obstacle n'est observé qu'à proximité, il faut D\* Lite (LPA\* + déplacement de l'agent, but et départ échangés) — piste naturelle pour un approfondissement ultérieur ;
- les coûts unitaires cachent le cas pondéré (terrain à $c(s, s')$ variable) — le mécanisme de `rhs` le supporte tel quel, mais les témoins seraient à re-mesurer.

In [8]:
# Test negatif de l'op 6 : couper la propagation = croire une valeur fausse.
# Monde a goulet : mur vertical x=12, ouverture unique en (12, 8), bords haut/bas ouverts.
W2, H2 = 24, 16
PONT = (12, 8)
mur_goulet = {(12, y) for y in range(1, H2 - 1)} - {PONT}

g_pont = Grid(W2, H2, mur_goulet)
c_avant, plan_pont = astar(g_pont, START, GOAL)
assert PONT in set(plan_pont), "le plan initial passe bien par le pont"
print(f"Goulet : plan initial cout {c_avant}, unique passage en {PONT}")

# (a) AUCUNE propagation : on meure le pont, l'etat interne ne bouge pas
lp_croire = LPAStar(Grid(W2, H2, mur_goulet), START, GOAL)
lp_croire.compute_shortest_path()
lp_croire.grid.walls.add(PONT)                        # le monde change, l'etat interne non
c_cru = lp_croire.g[GOAL]
print(f"(a) sans propagation : g(goal) affirme toujours {c_cru}")

# (b) reference honnete : A* from scratch apres blocage du pont
exp_goulet = [0]
c_vrai, _ = astar(Grid(W2, H2, mur_goulet | {PONT}), START, GOAL, exp_goulet)
print(f"(b) optimum reel apres blocage : {c_vrai} ({exp_goulet[0]} expansions scratch)")

# (c) propagation COMPLETE par LPA* : la meme instance, traitee honnetement
lp_vrai = LPAStar(Grid(W2, H2, mur_goulet), START, GOAL)
lp_vrai.compute_shortest_path()
e_avant_g = lp_vrai.expansions
lp_vrai.change_walls(added={PONT})
c_lpa_g = lp_vrai.compute_shortest_path()
e_repair_g = lp_vrai.expansions - e_avant_g
print(f"(c) LPA* repair : {c_lpa_g} en {e_repair_g} expansions "
      f"(vs {exp_goulet[0]} au scratch -- un changement structurel n'economise presque rien)")
print(f"Ecart de croyance : {c_cru} affirme vs {c_vrai} reel -> la garantie n'etait pas "
      f"deposee dans le calcul initial, elle est retablie par la propagation")

Goulet : plan initial cout 34, unique passage en (12, 8)
(a) sans propagation : g(goal) affirme toujours 34
(b) optimum reel apres blocage : 36 (276 expansions scratch)


(c) LPA* repair : 36 en 269 expansions (vs 276 au scratch -- un changement structurel n'economise presque rien)
Ecart de croyance : 34 affirme vs 36 reel -> la garantie n'etait pas deposee dans le calcul initial, elle est retablie par la propagation


### Lecture du résultat

Deux enseignements, le second n'étant pas le plus confortable :

- **La croyance fausse est mesurée** : sans propagation, $g(goal)$ affirme 34 alors que l'optimum réel est 36. Un état interne non propagé n'est pas « un peu optimiste » : il **ment**. On n'en extrait même pas de plan — un état incohérent n'a pas de chemin bien défini. La garantie d'optimalité n'est donc pas une propriété que le premier calcul aurait « déposée » quelque part : elle est **ré-établie à chaque changement par la propagation des incohérences** — les conditions de bord (jusqu'où propager) ne sont pas un détail d'implémentation, elles SONT la garantie.
- **La localité paie quand le changement est local** : ici le repair LPA\* coûte presque autant que le scratch (269 contre 276), car bloquer l'unique passage invalide *tout* ce qui est au-delà du pont. LPA\* ne dégénère jamais en pire que scratch, mais son avantage (23× sur le changement local du début) disparaît quand le changement est structurel. C'est la dette honnête de l'opération : la garantie est transportée, l'économie ne l'est que si le monde change localement.

## Exercices (C.1 — à compléter)

Les trois exercices reprennent l'instance du carnet (`BASE_WALLS`, `START`, `GOAL`, `blocked`). Chaque stub s'exécute sans erreur une fois complété ; les attendus ont été mesurés avec le code ci-dessus.

In [9]:
# Exercice 1 : reimplanter le coeur du repair -- _update_vertex.
# La classe LPAStarEtu est une copie ou _update_vertex est vide. Completez-la pour que
# le repair retrouve l'optimalite : rhs(u) = min sur les voisins entrants de g(s') + c(s', u),
# puis (re)insertion dans U uniquement si u reste incoherent.
class LPAStarEtu(LPAStar):
    def _update_vertex(self, u):
        pass  # TODO etudiant : recalculer rhs(u) puis maintenir la file U
        return None


lp_etu = LPAStarEtu(Grid(W, H, BASE_WALLS), START, GOAL)
lp_etu.compute_shortest_path()
lp_etu.change_walls(added={blocked})
c_etu = lp_etu.compute_shortest_path()
print(f"Exercice 1 a completer : repair via _update_vertex etudiant")
print(f"Attendu : cout {c_scratch} (optimal), repair 12 expansions ; obtenu : {c_etu}")

Exercice 1 a completer : repair via _update_vertex etudiant
Attendu : cout 34 (optimal), repair 12 expansions ; obtenu : inf


In [10]:
# Exercice 2 : faire croitre la dette du repair nave -- position du blocage.
# Bloquez la cellule plan_init[k] pour k dans {5, 10, 15, 20} et mesurez la dette
# (cout naif - optimal scratch) a chaque fois : ou l'obstacle fait-il le plus mal ?
dettes = {}
for k in (5, 10, 15, 20):
    pass  # TODO etudiant : bloquer plan_init[k], calculer naif et optimal, stocker la dette
print("Exercice 2 a completer : dette du repair nave selon la position du blocage")
print(f"Attendu : une dette non nulle sur au moins une position (mesure : "
      f"k=10 donne +{c_naif - c_scratch}) ; obtenu : {dettes}")

Exercice 2 a completer : dette du repair nave selon la position du blocage
Attendu : une dette non nulle sur au moins une position (mesure : k=10 donne +2) ; obtenu : {}


In [11]:
# Exercice 3 : la clause d'arret. Combien d'expansions le test k_top >= key(goal)
# evite-t-il ? Executez la propagation SANS la clause (propagez TOUTES les incoherences
# restantes de U) puis comparez le nombre d'expansions a celui du repair standard.
exp_avec_arret = None   # TODO etudiant : e_repair du carnet (ou re-mesurez-le)
exp_sans_arret = None   # TODO etudiant : compter les expansions en vidant U entierement
print("Exercice 3 a completer : expansions evitees par la clause d'arret")
print(f"Attendu : arret des le but coherent (12 expansions) vs vidage complet plus couteux")

Exercice 3 a completer : expansions evitees par la clause d'arret
Attendu : arret des le but coherent (12 expansions) vs vidage complet plus couteux


## Conclusion

**Ce carnet a attesté la seconde occurrence de l'opération 6 — « réparer localement sous garantie » — sur le substrat recherche de chemin**, avec les trois témoins exigés par la table :

1. **La garantie transportée** : coût LPA\* = coût A\* from scratch à chaque étape d'une séquence de 8 changements (8/8 concordances) ;
2. **La localité payée juste** : 12 expansions pour le repair contre 273 pour le ré-apprentissage complet (23×), budget cumulé minoré sur la séquence (9 contre 2258) ;
3. **Les contre-témoins** : le repair naïf perd la garantie (dette mesurée, chemin valide non optimal) ; l'absence de propagation rend l'état interne faux (plan traversant un mur, coût affirmé sans chemin réel).

La première attestation est le binôme [GT-13b/13c (Safe Subgame Solving)](../../GameTheory/) : réparer la stratégie dans un sous-jeu en préservant la non-exploitabilité. Deux substrats indépendants, même patron — l'opération 6 reste en table avec deux attestations.

**Position dans la série** : ce carnet approfondit [Search-03 (A\*)](Search-03-Informed.ipynb) — il suppose la file de priorité, l'heuristique admissible et la garantie d'optimalité d'A\* acquis. Il fait suite aux approfondissements [Search-11d (descente sous budget)](Search-11d-Descente-Sous-Budget.ipynb), [Search-12a (composer des regards)](Search-12a-Composer-Regards.ipynb) et [Search-13a (traverser des murs certifiés)](Search-13a-Traverser-Murs-Certifies.ipynb), seconds témoins des opérations 11, 12 et 13 de la [table des opérations (EPIC #12204)](https://github.com/jsboige/CoursIA/issues/12204).

**Référence** : Sven Koenig & Maxim Likhachev, *D\* Lite*, AAAI 2002 — la version déplacement (but fixe, départ mobile) de LPA\*, citée ici comme prolongement naturel pour l'observation partielle.

**Navigation** : [<< Search-03e — optimalité de A\*](Search-03e-AStar-Optimality.ipynb) | [Index](README.md) | [retour au parent Search-03 (A\*) ↑](Search-03-Informed.ipynb)
